# Validação de Qualidade de Dados — Tech Challenge Fase 2

Notebook de evidência das validações exigidas: duplicidade, valores ausentes,
consistência de chaves e integridade entre camadas (Bronze → Silver).

## 0. Setup

In [1]:
from google.cloud import bigquery

PROJECT_ID = "techchallenge2-afabetizacao"
client = bigquery.Client(project=PROJECT_ID)

def rodar(query):
    return list(client.query(query).result())

## 1. Duplicidade — Bronze UF
Chave esperada: `ano + sigla_uf + serie + rede`

In [2]:
resultado = rodar("""
    SELECT ano, sigla_uf, serie, rede, COUNT(*) AS ocorrencias
    FROM `techchallenge2-afabetizacao.bronze.uf`
    GROUP BY ano, sigla_uf, serie, rede
    HAVING COUNT(*) > 1
""")
print("OK - sem duplicidade" if len(resultado) == 0 else f"ATENCAO - {len(resultado)} duplicidades encontradas")
for linha in resultado:
    print(dict(linha))

OK - sem duplicidade


## 2. Duplicidade — Bronze Município
Chave esperada: `ano + id_municipio + serie + rede`

In [3]:
resultado = rodar("""
    SELECT ano, id_municipio, serie, rede, COUNT(*) AS ocorrencias
    FROM `techchallenge2-afabetizacao.bronze.municipio`
    GROUP BY ano, id_municipio, serie, rede
    HAVING COUNT(*) > 1
""")
print("OK - sem duplicidade" if len(resultado) == 0 else f"ATENCAO - {len(resultado)} duplicidades encontradas")
for linha in resultado:
    print(dict(linha))

OK - sem duplicidade


## 3. Duplicidade — Bronze Alunos
Chave esperada: `id_aluno + ano`

In [4]:
resultado = rodar("""
    SELECT id_aluno, ano, COUNT(*) AS ocorrencias
    FROM `techchallenge2-afabetizacao.bronze.alunos`
    GROUP BY id_aluno, ano
    HAVING COUNT(*) > 1
""")
print("OK - sem duplicidade" if len(resultado) == 0 else f"ATENCAO - {len(resultado)} duplicidades encontradas")
for linha in resultado:
    print(dict(linha))

OK - sem duplicidade


## 4. Valores Ausentes — Silver UF
Verificação de nulos em campos críticos para análise.

In [5]:
resultado = rodar("""
    SELECT
      COUNTIF(taxa_alfabetizacao IS NULL) AS nulos_taxa,
      COUNTIF(sigla_uf IS NULL) AS nulos_sigla_uf,
      COUNTIF(meta_alfabetizacao_2024 IS NULL) AS nulos_meta_2024,
      COUNT(*) AS total_linhas
    FROM `techchallenge2-afabetizacao.silver.uf`
""")
print(dict(resultado[0]))

{'nulos_taxa': 0, 'nulos_sigla_uf': 0, 'nulos_meta_2024': 3, 'total_linhas': 145}


## 5. Valores Ausentes — Silver Município
Verificação de nulos em campos críticos para análise.

In [6]:
resultado = rodar("""
    SELECT
      COUNTIF(taxa_alfabetizacao IS NULL) AS nulos_taxa,
      COUNTIF(id_municipio IS NULL) AS nulos_id_municipio,
      COUNTIF(meta_alfabetizacao_2024 IS NULL) AS nulos_meta_2024,
      COUNT(*) AS total_linhas
    FROM `techchallenge2-afabetizacao.silver.municipio`
""")
print(dict(resultado[0]))

{'nulos_taxa': 0, 'nulos_id_municipio': 0, 'nulos_meta_2024': 1312, 'total_linhas': 23995}


## 6. Validação de Chaves — Cobertura de UFs
Esperado: 27 UFs. Gap conhecido de DF/RR documentado no README.

In [7]:
resultado = rodar("SELECT COUNT(DISTINCT sigla_uf) AS total_ufs_presentes FROM `techchallenge2-afabetizacao.bronze.uf`")
total = resultado[0]['total_ufs_presentes']
print(f"UFs presentes: {total}/27" + (" - OK" if total == 27 else " - GAP CONHECIDO (DF/RR ausentes, ver README)"))

UFs presentes: 25/27 - GAP CONHECIDO (DF/RR ausentes, ver README)


## 7. Consistência entre Tabelas — UF x Meta Alfabetização UF
Verifica se toda UF presente na tabela `uf` possui meta correspondente.

In [8]:
resultado = rodar("""
    SELECT DISTINCT u.sigla_uf
    FROM `techchallenge2-afabetizacao.bronze.uf` u
    LEFT JOIN `techchallenge2-afabetizacao.bronze.meta_alfabetizacao_uf` m
      ON u.sigla_uf = m.sigla_uf AND u.ano = m.ano
    WHERE m.sigla_uf IS NULL
""")
print("OK - todas as UFs tem meta correspondente" if len(resultado) == 0 else f"ATENCAO - {len(resultado)} UFs sem meta")
for linha in resultado:
    print(dict(linha))

OK - todas as UFs tem meta correspondente


## 8. Consistência entre Tabelas — Município x Meta Alfabetização Município
Verifica se todo município presente possui meta correspondente.

In [9]:
resultado = rodar("""
    SELECT DISTINCT mu.id_municipio
    FROM `techchallenge2-afabetizacao.bronze.municipio` mu
    LEFT JOIN `techchallenge2-afabetizacao.bronze.meta_alfabetizacao_municipio` me
      ON mu.id_municipio = me.id_municipio AND mu.ano = me.ano
    WHERE me.id_municipio IS NULL
    LIMIT 20
""")
print(f"Municipios sem meta: {len(resultado)}" + (" - OK" if len(resultado) == 0 else " - ver README (limitacao de cobertura)"))

Municipios sem meta: 20 - ver README (limitacao de cobertura)


## 9. Validação Cruzada — Bronze vs Silver
Confirma que os LEFT JOINs da camada Silver não duplicaram linhas do Bronze.

In [10]:
resultado = rodar("""
    SELECT
      (SELECT COUNT(*) FROM `techchallenge2-afabetizacao.bronze.uf`) AS bronze_uf,
      (SELECT COUNT(*) FROM `techchallenge2-afabetizacao.silver.uf`) AS silver_uf,
      (SELECT COUNT(*) FROM `techchallenge2-afabetizacao.bronze.municipio`) AS bronze_municipio,
      (SELECT COUNT(*) FROM `techchallenge2-afabetizacao.silver.municipio`) AS silver_municipio
""")
r = dict(resultado[0])
print(r)
print("OK - contagens batem" if r['bronze_uf'] == r['silver_uf'] and r['bronze_municipio'] == r['silver_municipio'] else "ATENCAO - possivel duplicacao no JOIN")

{'bronze_uf': 145, 'silver_uf': 145, 'bronze_municipio': 23995, 'silver_municipio': 23995}
OK - contagens batem
